# Introduction to Nonlinear Data Structures 
In this module, we will take a deep dive into nonlinear data structures. We'll explore the differences between linear and nonlinear data structures, and then focus on dictionaries, sets, trees, and graphs, which are the four pillars of nonlinear structures that we’ll explore in this course.

By the end of this module, you will be able to explain and implement the following concepts:
* __Explain the differences between linear and nonlinear data structures__: We will discuss the key differences between linear and nonlinear data structures, including how they store and access data, as well as their performance characteristics.
* __Explain and implement the four pillars of nonlinear structures__: There are four main classes of nonlinear data structures: associative collections (e.g., dictionaries), uniqueness-based collections (e.g., sets), hierarchical models (e.g., trees), and general relationships (e.g., graphs). We'll explore each of these examples in detail.
* __Implement core traversal algorithms__: We will cover the different traversal algorithms for trees and graphs, including depth-first search (DFS) and breadth-first search (BFS). You will learn how to implement these algorithms and understand their applications.


Ready to tackle structures that map, connect, and hierarchize data? Let’s dive in!
___

## Linear vs. Nonlinear Data Structures
Linear data structures are organized in a sequential manner, where each element is connected to its previous and next elements. Examples include arrays, linked lists, stacks, and queues. We access elements in a linear fashion, one after another, which makes them suitable for tasks that require ordered data processing. 

On the other hand, nonlinear data structures:
* __Are not sequential__: Nonlinear data structures do not have a sequential arrangement. Instead, they allow for more complex relationships between elements. For example, in a tree structure, we have parent–child relationships where each parent node can have zero, one, or multiple children (e.g., a real-life family tree). In a graph, nodes can be connected in various ways without a strict order (e.g., a social network).
* __Associative & uniqueness-based access__: Instead of positional indexing, associative collections (dictionaries) map unique keys to values for constant-time lookup, while sets store only unique elements and support fast membership tests and set operations (union, intersection, difference).
* __Traversal__: Linear structures are traversed sequentially, e.g., using a for- or while-loop to access each element in order. In contrast, nonlinear data structures often require specialized traversal techniques to explore all elements—for instance, exploring branches of a tree before backtracking or following edges in a graph along various paths.

Let's dig into two classes of nonlinear data structures: associative collections, and Graphs (and trees).
___

## Associative and Uniqueness-based Collections
Associative collections, such as dictionaries, allow us to store key-value pairs where each key is unique. This enables fast lookups, insertions, and deletions based on the key. Uniqueness-based collections, such as sets, store only unique elements and provide efficient membership tests and set operations (union, intersection, and difference).

### Dictionaries
A dictionary `D{K, V}` associates each key of type `K` with a value of type `V`, enabling average-case constant-time $\mathcal{O}(1)$ lookup, insertion, and deletion.
* _Secret sauce_: Under the hood, dictionaries use **hashing** to turn each key into an index into a fixed-size array (whose elements are called buckets). When we look up a key, the hash function is applied to find the index and retrieve the value. Thus, dictionaries are implemented as arrays with a hashing mechanism!

Let's look at the classic djb2 (or `times 33`) string‐hash algorithm, originally written by Daniel J. Bernstein. This is a simplified version of what happens in a dictionary:

__Initialize__: Given a string $S$, a table of size $N$, a seed $R\gets{5381}$. Set $\text{index}\gets{0}$.

1. Compute the length of the string: $L \gets \text{len}(S)$
2. Compute the integer values of the character array: $C \gets [\texttt{CodePoint}(c)\;\big|\;c \in S]$
3. For $i = {0}\;\text{to}\; L-1$ __do__:
   - Set $R\gets \left[(R << 5) + R\right] + C_{i}$ 
4. Return the index: $\text{index} \gets (R \mod N + N) \mod N$.

Let's implement this logic, test it with a few example strings, and then walk through what is going on.

In [10]:
index, teststring = let

    # initialize -
    string_to_hash = "This is a test string";
    L = length(string_to_hash);
    C = collect(string_to_hash) .|> x -> Int(x); # convert to Int
    R = 5381; # a large prime number
    N = 10000; # number of buckets in the hash table

    # hash function -
    for i ∈ 1:L
        R = (R << 5) + R + C[i]; # << 5 is a left shift operation by 5 bits, 
    end
    index = (R % N + N) % N;

    index, string_to_hash # return
end

(7329, "This is a test string")

#### What is a collision?
A collision occurs when two different keys hash to the same index in the dictionary. This can happen when the hash function produces the same index for different keys. How we respond to collisions is next level, but it's pretty easy to see how this can happen. 
* __Intuition__: if we have two keys that are very similar, like `cat` and `mat`, and a small number of buckets in the hash table, they are likely to collide. There is just not enough space to store both keys in the hash table without them pointing to the same index.

Let's see what happens when we hash the strings `cat` and `mat` with a small number of buckets (N = 10) in the hash table.

In [14]:
index, teststring = let

    # initialize -
    string_to_hash = "mat"; # try mat - what happens?
    L = length(string_to_hash);
    C = collect(string_to_hash) .|> x -> Int(x); # convert to Int
    R = 5381; # a large prime number
    N = 10; # number of buckets in the hash table (make it small to force collisions)

    # hash function -
    for i ∈ 1:L
        R = (R << 5) + R + C[i]; # << 5 is a left shift operation by 5 bits, 
    end
    index = (R % N + N) % N;

    index, string_to_hash # return
end

(5, "mat")

### Sets
Now that we understand dictionaries, we can explore sets. Sets leverage the same hash-table machinery as dictionaries, but discard the `value` and only track keys.

* __Unique elements__: When you add an element, its hash code points to a slot. If the slot is empty, the element is stored; if it already contains that key, the insertion is skipped, ensuring uniqueness. Of course, this assumes that the hash function is well-designed to minimize collisions.
* __Membership tests__: To test membership, the element’s hash code points to a slot; if that key occupies it, the element is in the set. This runs in $\mathcal{O}(1)$ time.  
* __Under the hood__: In Julia, `Set{T}` is a thin wrapper around `Dict{T, Nothing}`; in Python, `set` is a standalone C-level hash table that holds only keys. Both reuse the same hash functions, bucket arrays, and collision-resolution logic to deliver ℴ(1) performance for inserts, deletes, and lookups.

___

<div>
    <center>
      <img
        src="figs/Fig-Graphs-Directed-Undirected.svg"
        alt="General Tree Example"
        height="400"
        width="800"
      />
    </center>
  </div>

## Graphs
A **simple graph** $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ consists of a set of vertices $\mathcal{V}$ and a set of edges $\mathcal{E}$ connecting pairs of vertices $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$. In a __simple graph__ there are no self-loops (edges connecting a vertex to itself) or parallel edges (multiple edges connecting the same pair of vertices).

Edges may carry additional structure:
- **Weight**: In an _unweighted_ graph, all edges are treated equally, while in a _weighted_ graph each edge $e\in\mathcal{E}$ has a numerical weight $w(e)$ that can represent cost, distance, capacity, etc. This allows for more complex relationships and optimizations in graph algorithms.
- **Direction**:  In an _undirected_ graph, edges are bidirectional, meaning the connection between vertices is symmetric. In a _directed_ graph, edges have a direction, indicating a one-way relationship from one vertex to another. Thus, in an undirected graph $\{u,v\} = \{v,u\}$, modeling symmetric relationships. However, for a directed graph, $(u,v) \neq (v,u)$, modeling asymmetric relationships or dependencies. 

A graph can also be classified based on its properties:
- **Cyclic vs. Acyclic**: A graph is _cyclic_ if it contains at least one cycle (a closed path returning to its start).  It is _acyclic_ if no such cycle exists. (Acyclic undirected graphs are called _forests_; acyclic directed graphs are _DAGs_.)
- **Connectedness**:  In an _undirected_ graph, _connected_ means there is a path between every pair of vertices; otherwise it is _disconnected_. In a _directed_ graph, _strongly connected_ requires a directed path both ways between every pair, while _weakly connected_ ignores edge orientation.

### Graph Representations
An __adjacency matrix__ $\mathcal{A}$ representation of a graph $\mathcal{G} = (\mathcal{V},\mathcal{E})$ is a $\dim\mathcal{V}\times\dim\mathcal{V}$ matrix holding integer or floating point values. The entry for row $i$ and column $j$ of the matrix $\mathcal{A}$, denoted $a_{ij}\in\mathcal{A}$, describes the connection between vertices $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$. 
* __Unweighted__: If there is an edge connecting $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$ and graph $\mathcal{G}$ is unweighted, then $a_{ij}=1$ (true) , otherwise $a_{ij}=0$ (false).
* __Weighted__: In cases where the edges of graph $\mathcal{G}$ have weights, if there is an edge connecting $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$, then $a_{ij}=w_{ij}$, otherwise $a_{ij}=0$ (false), where $w_{ij}\in\mathbb{R}$ denotes the weight of the edge connecting $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$.

An __adjacency list__ for a graph $\mathcal{G} = (\mathcal{V},\mathcal{E})$ is a $\dim\mathcal{V}$ dictionary $d$, where
the ith entry points to the children indices of vertex $v_{i}\in\mathcal{V}$, denoted by set $\mathcal{C}_{i}$. 
In other words, $d_{i}\rightarrow\mathcal{C}_{i}$. There is a single entry in dictionary $d$ for each vertex in graph $\mathcal{G}$.

____

<div>
    <center>
      <img
        src="figs/Fig-General-Tree-Example.svg"
        alt="General Tree Example"
        height="400"
        width="800"
      />
    </center>
  </div>

## Trees
A __rooted tree__ $T = \left\{\mathcal{V},\mathcal{E}\right\}$ composed of nodes $\mathcal{V}$ and edges $\mathcal{E}$, is a connected, acyclic graph with a  **root** node and a parent–child hierarchy. Each node has at most one parent and zero or more children, making trees ideal for modeling hierarchical data (e.g., file systems, org charts, etc). Let's look at some key elements of a tree structure:
* **Levels and Height**: Each node in the tree is assigned a level based on its distance from the root (node with no parent). The root is at level $h=0$, its children are at $h=1$, and so on. For any node, the number of edges on the path from the root to that node defines its level. The maximum level $\max_{v\in\mathcal{V}}\left(\texttt{level}(v)\right)$ of a tree is called the height $h$, here `h = 3`.
* **Leaves and Branches**: Leaves are nodes with no children; they can occur at any level of a tree. A branch is any path from the root down through its descendants to a leaf. One such root-to-leaf path is highlighted to illustrate how height corresponds to the length of the longest branch in the tree.

Trees are widely used for various applications:
* __Binary Price Tree__: A tree where each node has at most two children, referred to as the left and right child. This structure is widely used, for example, in finance to represent future prices of an asset, where each node represents a possible price at a given time, and the left child represents a lower price while the right child represents a higher price.
* __Binary Search Tree (BST)__: A binary tree where the left child is less than the parent node, and the right child is greater than the parent node. This property allows for efficient searching, insertion, and deletion operations.
* __Heaps__: A specialized tree structure that satisfies the heap property, where the parent node is either greater than or equal to (max heap) or less than or equal to (min heap) its children. Heaps are commonly used in priority queues and sorting algorithms.
* __Tries__: A tree-like structure used for storing strings, where each node represents a character in the string. Tries are particularly useful for tasks such as autocomplete and spell checking, as they enable efficient prefix searching and retrieval of strings.

Wow! Trees are just connected acyclic graphs that have a hierarchical structure. Thus, all the concepts we learned about graphs (including some stuff to come) also apply to trees. 
___

## Traversal Algorithms
We visit (traverse) the nodes in a graph (or tree) differently than in an array. Let's look at two common traversal algorithms for graphs (that we'll use as building blocks for other algorithms): depth-first search (DFS) and breadth-first search (BFS).

### Depth-First Search (DFS)
Depth-first search (DFS) __recursively__ explores as far as possible along each branch before backtracking. It uses a `Set` data structure to keep track of the nodes that have already been visited. 

The algorithm starts at a given node, marks it as visited, and then recursively visits each unvisited neighbor until all reachable nodes are visited. If a node has no unvisited neighbors, the algorithm backtracks to the previous node and continues exploring from there. Let's look at a simple recursive implementation of a DFS algorithm.

__Initialization__: Given a graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$, a starting vertex $v_{s}\in\mathcal{V}$, and an empty set of visited vertices $\mathcal{V}_{\text{visited}}$.

1. If $v_{s}\notin\mathcal{V}_{\text{visited}}$, then:
    - Add $v_{s}$ to the $\mathcal{V}_{\text{visited}}$ set: $\mathcal{V}_{\text{visited}}\gets\mathcal{V}_{\text{visited}}\cup\{v_{s}\}$.
    - Get the neighbors of node $v_{s}$: Set $\mathcal{N}_{s} \gets \texttt{neighbors}(v_{s})$.
    - For each neighbor $v_{n}\in\mathcal{N}_{s}$, do:
        - __Recursively__ call the DFS algorithm with $v_{n}$ as the new starting vertex. (Goto step 1 with $v_{n}$ as the new starting vertex.)
2. If $v_{s}\in\mathcal{V}_{\text{visited}}$, then return.


DFS runs in $\mathcal{O}(|\mathcal{V}|+|\mathcal{E}|)$ time, where $|\mathcal{V}|$ is the number of vertices and $|\mathcal{E}|$ is the number of edges in the graph. It uses $\mathcal{O}(|\mathcal{V}|)$ space for the visited set (plus recursion depth).


### Breadth-First Search (BFS)
Breadth-first search (BFS) is a traversal algorithm that explores all the neighbors of a node before moving on to the next level of nodes. It uses a `Queue` data structure to keep track of the nodes to visit next. 

The algorithm starts at a given node, marks it as visited, and then enqueues all its unvisited neighbors. It continues to dequeue nodes from the front of the queue, marking them as visited and enqueuing their unvisited neighbors, until all reachable nodes are visited. Let's look at an implementation of a BFS algorithm:

__Initialization__: Given a graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$, a starting vertex $v_{s}\in\mathcal{V}$, and an empty Set of visited vertices $\mathcal{V}_{\text{visited}}$, and an empty queue $\mathcal{Q}$.

1. Add the starting vertex $v_{s}$ to the queue: $\mathcal{Q}\gets\texttt{enqueue}(\mathcal{Q}, v_{s})$.
2. While the queue $\mathcal{Q}$ is not empty, __do__:
    - Dequeue a vertex $v_{n}$ from the front of the queue: $v_{n}\gets\texttt{dequeue}(\mathcal{Q})$.
    - If $v_{n}\notin\mathcal{V}_{\text{visited}}$, __then__:
        - Add $v_{n}$ to the $\mathcal{V}_{\text{visited}}$ set: $\mathcal{V}_{\text{visited}}\gets\mathcal{V}_{\text{visited}}\cup\{v_{n}\}$.
        - Get the neighbors of node $v_{n}$: Set $\mathcal{N}_{n} \gets \texttt{neighbors}(v_{n})$.
        - For each neighbor $v_{m}\in\mathcal{N}_{n}$, do:
            - If $v_{m}\notin\mathcal{V}_{\text{visited}}$, then enqueue it: $\mathcal{Q}\gets\texttt{enqueue}(\mathcal{Q}, v_{m})$.

BFS runs in $\mathcal{O}(|\mathcal{V}|+|\mathcal{E}|)$ time, where $|\mathcal{V}|$ is the number of vertices and $|\mathcal{E}|$ is the number of edges in the graph. It uses $\mathcal{O}(|\mathcal{V}|)$ space for the visited set (plus queue space).

____